In [1]:
import numpy as np
from numpy.random import randn
import random

In [29]:
train_data = {
    'хороший': True,
    'круто': True,
    'весело': True,
    'ахуенно': True,
    'ултра хорош': True,
    'супер мега хорош': True,
    'неплохо': True,
    'не грустно': True,
    'очень хорошо': True,
    'счаслив': True,
    'я хорош': True,
    'я охуителен': True,
    'мне хорошо': True,
    'мне не плохо': True,
    'Это очень хорошо': True,
    'хуево': False,
    'насрал': False,
    'плохо': False,
    'не хорошо': False,
    'кринжанул': False,
    'ультра насрал': False,
    'не радует': False,
    'радует': True,
    'грустно': False,
    'весело': True,
}

test_data = {
  'это счастье': True,
  'я хороший': True,
  'я не хороший': False,
  'мне не грустно': True,
  'я очень хорош': True,
  'мне очень грустно': False,
  'это плохо, а не хорошо': False,
}

In [3]:
vocab = list(set([w for text in train_data.keys() for w in text.split(' ')]))
vocab_size = len(vocab)

print('%d unique words found' % vocab_size)

23 unique words found


In [4]:
word_to_idx = { w: i for i, w in enumerate(vocab) }
idx_to_word = { i: w for i, w in enumerate(vocab) }

In [5]:
def createInputs(text):
    inputs = []
    for w in text.split(' '):
        v = np.zeros((vocab_size, 1))
        v[word_to_idx[w]] = 1
        inputs.append(v)
    
    return inputs
def softmax(xs):
    return np.exp(xs) / sum(np.exp(xs))


In [10]:
class RNN:
    def __init__(self, input_size, output_size, hidden_size=64):
        #веса
        self.Whh = randn(hidden_size ,hidden_size)/ 1000
        self.Wxh = randn(hidden_size ,input_size)/ 1000
        self.Why = randn(output_size ,hidden_size)/ 1000
        #микровеса
        self.bh = np.zeros((hidden_size , 1))
        self.by = np.zeros((output_size , 1))
        
    def forward(self, inputs):
        h = np.zeros((self.Whh.shape[0], 1))
        
        self.last_input = inputs
        self.last_hs  = {0: h} 
        
        for i, x in enumerate(inputs):
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            self.last_hs[i + 1] = h 
            
        y = self.Why @ h + self.by
        return y ,h 
    def backpros(self , d_y, learn_rate=2e-1):
        n = len(self.last_input)
        
        d_Why = d_y @ self.last_hs[n].T
        d_by = d_y 
        
        d_Whh = np.zeros(self.Whh.shape)
        d_Wxh = np.zeros(self.Wxh.shape)
        d_bh = np.zeros(self.bh.shape)
        
        d_h =  self.Why.T @ d_y
        
        for t in reversed(range(n)):
            temp = ((1-self.last_hs[t + 1] ** 2) * d_h)
            
            d_bh += temp
            d_Whh += temp @ self.last_hs[t].T
            d_Wxh += temp @ self.last_input[t].T
            
            d_h = self.Whh @ temp
        
        for d in [d_Wxh, d_Whh, d_Why, d_bh, d_by]:
            np.clip(d, -1, 1 ,out =d)
            
        self.Whh -= learn_rate * d_Whh
        self.Wxh -= learn_rate * d_Wxh
        self.Why -= learn_rate * d_Why
        self.bh -= learn_rate * d_bh
        self.by -= learn_rate * d_by

In [11]:
rnn = RNN(vocab_size, 2)

In [14]:
def processData(data ,backprop = True):
    items = list(data.items())
    random.shuffle(items)
    
    loss = 0 
    num_correct = 0 
     
    for x ,y in items:
        inputs = createInputs(x)
        target = int(y)
        
        out, _ = rnn.forward(inputs)
        probs = softmax(out)
        
        loss -= np.log(probs[target])
        num_correct += int(np.argmax(probs) == target)
        
        if backprop:
            d_L_d_y = probs
            d_L_d_y[target] -= 1 
            
            rnn.backpros(d_L_d_y)
    return loss / len(data) , num_correct / len(data)

Дальше не простой RNN. Сеть из двух LSTM, `lstm1` и `lstm2`. Модуль написан руками: вход, забывание, память и выход. Готового LSTM тогда не было.

Позитив учится на фразах с `True`, негатив на фразах с `False`. Класс 1 это позитив, класс 0 это негатив. Незнакомое слово идет в `<unk>`.


In [10]:
import re

def tokens(text):
    return re.findall(r"[a-zа-я0-9]+", text.lower().replace("ё", "е"))

def sigmoid(value):
    return 1 / (1 + np.exp(-np.clip(value, -20, 20)))

def softmax(xs):
    z = xs - np.max(xs)
    e = np.exp(z)
    return e / np.sum(e)

class LSTM:
    def __init__(self, input_size, hidden_size):
        scale = 1 / np.sqrt(input_size + hidden_size)
        self.hidden_size = hidden_size
        self.Wx = randn(4 * hidden_size, input_size) * scale
        self.Wh = randn(4 * hidden_size, hidden_size) * scale
        self.b = np.zeros((4 * hidden_size, 1))
        self.b[hidden_size:2 * hidden_size] = 1

    def step(self, x, h, c):
        z = self.Wx @ x + self.Wh @ h + self.b
        hsz = self.hidden_size
        i = sigmoid(z[0:hsz])
        f = sigmoid(z[hsz:2 * hsz])
        g = np.tanh(z[2 * hsz:3 * hsz])
        o = sigmoid(z[3 * hsz:4 * hsz])
        c_new = f * c + i * g
        h_new = o * np.tanh(c_new)
        return h_new, c_new, (x, h, c, i, f, g, o, c_new, h_new)

class LSTM2:
    def __init__(self, input_size, output_size, hidden_size=32):
        self.hidden_size = hidden_size
        self.lstm1 = LSTM(input_size, hidden_size)
        self.lstm2 = LSTM(hidden_size, hidden_size)
        self.layers = (self.lstm1, self.lstm2)
        self.Why = randn(output_size, hidden_size) / np.sqrt(hidden_size)
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs):
        h = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        c = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        self.cache = []
        for x in inputs:
            step_cache = []
            current = x
            for index, layer in enumerate(self.layers):
                h[index], c[index], cached = layer.step(current, h[index], c[index])
                step_cache.append(cached)
                current = h[index]
            self.cache.append(step_cache)
        self.last_h = h[1]
        return self.Why @ self.last_h + self.by

    def backward(self, d_y, learn_rate=0.15):
        d_Why = d_y @ self.last_h.T
        d_by = np.array(d_y, copy=True)
        grads = []
        for layer in self.layers:
            grads.append({
                "Wx": np.zeros_like(layer.Wx),
                "Wh": np.zeros_like(layer.Wh),
                "b": np.zeros_like(layer.b),
            })
        dh_next = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        dc_next = [np.zeros((self.hidden_size, 1)) for _ in self.layers]
        for t in reversed(range(len(self.cache))):
            dh = [item.copy() for item in dh_next]
            dc = [item.copy() for item in dc_next]
            if t == len(self.cache) - 1:
                dh[1] = dh[1] + self.Why.T @ d_y
            for li in (1, 0):
                layer = self.layers[li]
                x, h_prev, c_prev, i, f, g, o, c_new, h_new = self.cache[t][li]
                tanh_c = np.tanh(c_new)
                d_c = dh[li] * o * (1 - tanh_c ** 2) + dc[li]
                d_o = dh[li] * tanh_c
                d_i = d_c * g
                d_f = d_c * c_prev
                d_g = d_c * i
                d_z = np.concatenate([
                    d_i * i * (1 - i),
                    d_f * f * (1 - f),
                    d_g * (1 - g ** 2),
                    d_o * o * (1 - o),
                ], axis=0)
                grads[li]["Wx"] += d_z @ x.T
                grads[li]["Wh"] += d_z @ h_prev.T
                grads[li]["b"] += d_z
                dx = layer.Wx.T @ d_z
                dh_next[li] = layer.Wh.T @ d_z
                dc_next[li] = d_c * f
                if li == 1:
                    dh[0] = dh[0] + dx
        for grad in grads:
            for key in grad:
                np.clip(grad[key], -5, 5, out=grad[key])
        np.clip(d_Why, -5, 5, out=d_Why)
        np.clip(d_by, -5, 5, out=d_by)
        for layer, grad in zip(self.layers, grads):
            layer.Wx -= learn_rate * grad["Wx"]
            layer.Wh -= learn_rate * grad["Wh"]
            layer.b -= learn_rate * grad["b"]
        self.Why -= learn_rate * d_Why
        self.by -= learn_rate * d_by

unk = "<unk>"
vocab = [unk] + sorted({word for text in train_data for word in tokens(text)})
word_to_idx = {word: index for index, word in enumerate(vocab)}
vocab_size = len(vocab)
print("vocab %d, two LSTM layers" % vocab_size)

def createInputs(text):
    words = tokens(text) or [unk]
    inputs = []
    for word in words:
        vector = np.zeros((vocab_size, 1))
        vector[word_to_idx.get(word, 0)] = 1
        inputs.append(vector)
    return inputs

np.random.seed(1)
random.seed(1)
lstm = LSTM2(vocab_size, 2)

def processData(data, backprop=True):
    items = list(data.items())
    random.shuffle(items)
    loss = 0.0
    num_correct = 0
    for text, label in items:
        target = int(label)
        out = lstm.forward(createInputs(text))
        probs = softmax(out)
        loss -= float(np.ravel(np.log(probs[target] + 1e-9))[0])
        num_correct += int(np.argmax(probs) == target)
        if backprop:
            d_y = probs.copy()
            d_y[target] -= 1
            lstm.backward(d_y)
    return loss / len(data), num_correct / len(data)

def predict_label(text):
    probs = softmax(lstm.forward(createInputs(text))).ravel()
    name = "позитив" if int(np.argmax(probs)) == 1 else "негатив"
    return name, float(probs[1])

for epoch in range(120):
    train_loss, train_acc = processData(train_data)
    if epoch % 40 == 39:
        test_loss, test_acc = processData(test_data, backprop=False)
        print("--- epoch %d" % (epoch + 1))
        print("Train loss %.4f | acc %.2f" % (train_loss, train_acc))
        print("Test  loss %.4f | acc %.2f" % (test_loss, test_acc))

print()
for text, label in train_data.items():
    name, score = predict_label(text)
    expected = "позитив" if label else "негатив"
    print("train %-22s %s -> %s %.2f" % (text, expected, name, score))
print()
for text, label in test_data.items():
    name, score = predict_label(text)
    expected = "позитив" if label else "негатив"
    print("test  %-22s %s -> %s %.2f" % (text, expected, name, score))


vocab 25, two LSTM layers
--- epoch 40
Train loss 0.2743 | acc 0.88
Test  loss 2.1267 | acc 0.57
--- epoch 80
Train loss 0.0323 | acc 1.00
Test  loss 3.4861 | acc 0.57
--- epoch 120
Train loss 0.0033 | acc 1.00
Test  loss 4.7844 | acc 0.57

train хороший                позитив -> позитив 1.00
train круто                  позитив -> позитив 1.00
train весело                 позитив -> позитив 1.00
train ахуенно                позитив -> позитив 1.00
train ултра хорош            позитив -> позитив 1.00
train супер мега хорош       позитив -> позитив 1.00
train неплохо                позитив -> позитив 1.00
train не грустно             позитив -> позитив 0.98
train очень хорошо           позитив -> позитив 1.00
train счаслив                позитив -> позитив 1.00
train я хорош                позитив -> позитив 1.00
train я охуителен            позитив -> позитив 1.00
train мне хорошо             позитив -> позитив 1.00
train мне не плохо           позитив -> позитив 1.00
train Это очень х

На train позитивная фраза остается позитивом, негативная негативом.

На test часть фраз с `не` все еще идет за последним словом.


Вектор настроения из двух осей.

- ось 0: `0` злой, `1` добрый
- ось 1: `0` грустный, `1` веселый

`None` значит, что эту ось на фразе не учим. `злой и грустный` это `(0, 0)`, `добрый но грустный` это `(1, 0)`, `злой но веселый` это `(0, 1)`.


In [13]:
# ось 0: 0 злой, 1 добрый
# ось 1: 0 грустный, 1 веселый
# None значит эту ось на фразе не учим
vector_train = {
    "добрый": (1, None),
    "злой": (0, None),
    "веселый": (None, 1),
    "грустный": (1, 0),
    "круто": (1, 1),
    "радует": (1, 1),
    "ахуенно": (1, 1),
    "люблю": (1, 1),
    "мне хорошо": (1, 1),
    "счастье": (1, 1),
    "мне весело": (1, 1),
    "тоска": (1, 0),
    "печально": (1, 0),
    "мне грустно": (1, 0),
    "хуево": (0, 0),
    "плохо": (0, 0),
    "ненавижу": (0, 0),
    "бесит": (0, 0),
    "мне плохо": (0, 0),
    "добрый и веселый": (1, 1),
    "добрый и грустный": (1, 0),
    "злой и веселый": (0, 1),
    "злой и грустный": (0, 0),
    "добрый но грустный": (1, 0),
    "злой но веселый": (0, 1),
}

vector_vocab = ["<unk>"] + sorted({word for text in vector_train for word in tokens(text)})
vector_index = {word: index for index, word in enumerate(vector_vocab)}
vector_size = len(vector_vocab)

def vectorInputs(text):
    words = tokens(text) or ["<unk>"]
    inputs = []
    for word in words:
        vector = np.zeros((vector_size, 1))
        vector[vector_index.get(word, 0)] = 1
        inputs.append(vector)
    return inputs

np.random.seed(2)
random.seed(2)
vector_lstm = LSTM2(vector_size, 2)

def processVector(data, backprop=True):
    items = list(data.items())
    random.shuffle(items)
    loss = 0.0
    counted = 0
    correct = 0
    for text, target in items:
        out = vector_lstm.forward(vectorInputs(text))
        probs = sigmoid(out)
        grad = np.zeros((2, 1))
        ok = True
        used = False
        for axis, bit in enumerate(target):
            if bit is None:
                continue
            used = True
            counted += 1
            pred = float(probs[axis, 0])
            truth = float(bit)
            loss -= truth * np.log(pred + 1e-9) + (1 - truth) * np.log(1 - pred + 1e-9)
            grad[axis, 0] = pred - truth
            ok = ok and ((pred >= 0.5) == (truth >= 0.5))
        correct += int(used and ok)
        if backprop:
            vector_lstm.backward(grad)
    return loss / max(counted, 1), correct / len(data)

for epoch in range(160):
    train_loss, train_acc = processVector(vector_train)
    if epoch % 40 == 39:
        print("--- epoch %d  loss %.4f | acc %.2f" % (epoch + 1, train_loss, train_acc))

def mood_vector(text):
    kind, fun = sigmoid(vector_lstm.forward(vectorInputs(text))).ravel()
    kind_name = "добрый" if kind >= 0.5 else "злой"
    fun_name = "веселый" if fun >= 0.5 else "грустный"
    return kind_name, float(kind), fun_name, float(fun)

print()
for text in (
    "добрый",
    "злой",
    "грустный",
    "веселый",
    "круто",
    "хуево",
    "злой и грустный",
    "добрый но грустный",
    "злой но веселый",
    "ты добрый и веселый",
    "очень злой и грустный",
):
    kind_name, kind, fun_name, fun = mood_vector(text)
    print("%-24s %s %.2f | %s %.2f   vector=(%.2f, %.2f)" % (
        text, kind_name, kind, fun_name, fun, kind, fun
    ))


--- epoch 40  loss 0.1921 | acc 1.00
--- epoch 80  loss 0.0047 | acc 1.00
--- epoch 120  loss 0.0020 | acc 1.00
--- epoch 160  loss 0.0013 | acc 1.00

добрый                   добрый 1.00 | грустный 0.41   vector=(1.00, 0.41)
злой                     злой 0.00 | грустный 0.08   vector=(0.00, 0.08)
грустный                 добрый 1.00 | грустный 0.00   vector=(1.00, 0.00)
веселый                  добрый 0.99 | веселый 1.00   vector=(0.99, 1.00)
круто                    добрый 1.00 | веселый 1.00   vector=(1.00, 1.00)
хуево                    злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
злой и грустный          злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
добрый но грустный       добрый 1.00 | грустный 0.00   vector=(1.00, 0.00)
злой но веселый          злой 0.00 | веселый 1.00   vector=(0.00, 1.00)
ты добрый и веселый      добрый 1.00 | веселый 1.00   vector=(1.00, 1.00)
очень злой и грустный    злой 0.00 | грустный 0.00   vector=(0.00, 0.00)
